# Sprint E5 walkthrough: the risk model evaluation and the champion decision

In [1]:
# the repository root is importable so the package and the dashboard module can
# be imported from the notebook's working directory
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from efb import cov, eval_risk, evaluate, race, registry  # noqa: E402

DATA = ROOT / "data"
RESULTS = json.loads((ROOT / "sprints" / "E5" / "RESULTS.json").read_text())
REGISTRY = json.loads((DATA / "models" / "registry.json").read_text())
STORED = {name: block["stored_numbers"] for name, block in RESULTS["criteria"].items()}
VERDICTS = {name: block["verdict"] for name, block in RESULTS["criteria"].items()}
print("criteria loaded:", sorted(STORED))
print("versions registered:", sorted(REGISTRY["models"]))
print("champion:", registry.champion(REGISTRY))

criteria loaded: ['F5.1', 'F5.2', 'F5.3', 'F5.4', 'F5.5']
versions registered: ['PCA-v1', 'PCA-v1c', 'TS-v1', 'XS-v1', 'XS-v2']
champion: XS-v1


In [2]:
# Cell 1 rule: the data hash in the results file must be the hash of the
# artifacts this notebook reads, recomputed here from the files on disk. The
# value is never typed: it is compared against the stored one, so a rebuild
# that moves it moves both sides of the assertion together.
recomputed = evaluate.e5_data_hash(DATA)
stored_hash = str(RESULTS["data_hash"])
print("data_hash   ", stored_hash)
print("recomputed  ", recomputed)
assert stored_hash == recomputed, "the results file and the artifacts disagree"
assert len(stored_hash) == 64, "the stored hash is not a sha256"
print("e5 data hash asserted")

data_hash    210769d6ec92a325ec2f5553875ea4de0de29bf52051a527f69ab674c098b293
recomputed   210769d6ec92a325ec2f5553875ea4de0de29bf52051a527f69ab674c098b293
e5 data hash asserted


## 1. Every criterion, its stored number and its verdict

In [3]:
# the criterion text is printed as stored, so a reworded threshold would be
# visible right here rather than buried in a code change
for name, block in RESULTS["criteria"].items():
    print(name, "[" + block["verdict"] + "]")
    print("  criterion:", block["criterion"])
    print("  threshold:", block["threshold"])
    print("  stored_numbers:", json.dumps(block["stored_numbers"], sort_keys=True))

F5.1 [fail]
  criterion: At least one model version achieves mean bias between 0.9 and 1.1 across all portfolio families.
  threshold: 0.9 <= mean bias <= 1.1 for every family for at least one version
  stored_numbers: {"family_bias": {"PCA-v1": 1.1388412938997308, "PCA-v1c": 1.1391854728617519, "XS-v1": 1.0469839503253322, "XS-v2": 1.0384709489361823}, "inside_all_families": {"PCA-v1": false, "PCA-v1c": false, "XS-v1": false, "XS-v2": false}}
F5.2 [fail]
  criterion: Factor-based models beat the sample covariance on bias for long/short portfolios.
  threshold: every factor version's long/short mean bias closer to 1 than the sample covariance's
  stored_numbers: {"long_short_abs_bias_minus_1": {"pca_v1": 0.07288545487816167, "pca_v1c": 0.07545776635627166, "ts_v1": 0.06632785175621136, "xs_v1": 0.018568115349799186, "xs_v2": 0.02563794967801525}, "sample_long_short_abs_bias_minus_1": 0.07522700661959725}
F5.3 [pass]
  criterion: Bias is worst in 2020 Q1 for every model. This is expecte

## 2. By hand: the standardized return and the bias statistic

In [4]:
# one portfolio-day, recomputed from the raw inputs: z_t = r_t / sigma_hat
# where r_t uses the weights held from the previous close and sigma_hat is the
# xs_v1 forecast at the latest rebalance date at or before t-1
stored = pd.read_parquet(DATA / "eval" / "bias_xs_v1_long_only.parquet")
portfolios = pd.read_parquet(DATA / "eval" / "e5_portfolios.parquet")
forecasts = pd.read_parquet(DATA / "eval" / "e5_forecast_portfolios.parquet")
wide, _ = eval_risk.load_clean_wide(DATA)

portfolio = "long_only_00"
series = stored.loc[stored["portfolio"] == portfolio].sort_values("date")
day = series.loc[series["z"].notna()].iloc[100]
t = pd.Timestamp(day["date"])
rebalances = sorted(
    pd.to_datetime(
        portfolios.loc[portfolios["portfolio"] == portfolio, "date"].unique()
    )
)
stamp = max(rb for rb in rebalances if rb < t)
weights = (
    portfolios.loc[
        (portfolios["portfolio"] == portfolio)
        & (pd.to_datetime(portfolios["date"]) == stamp)
    ]
    .set_index("ticker")["weight"]
    .reindex(wide.columns)
    .fillna(0.0)
)
# the engine's missing-data semantics: an unpriced name contributes zero and a
# held name with a stale day makes the day missing, instead of 0 x NaN
r_t = float(
    eval_risk._portfolio_returns(
        weights.to_numpy(dtype=float)[None, :],
        wide.loc[t].to_numpy(dtype=float)[None, :],
    )[0, 0]
)
sigma = forecasts.loc[
    (forecasts["version"] == "xs_v1")
    & (forecasts["date"] == stamp)
    & (forecasts["portfolio"] == portfolio),
    "sigma",
].iloc[0]
z = r_t / float(sigma)
print("by hand z", round(z, 6), "stored z", round(float(day["z"]), 6))
assert abs(z - float(day["z"])) < 1e-9, "the stored z was not reproduced"

# B = sqrt(mean z^2) with the delta-method band SE(B) = sqrt(2/T)/(2B)
z_series = series["z"].to_numpy(dtype=float)
z_series = z_series[np.isfinite(z_series)]
bias = float(np.sqrt(np.mean(z_series**2)))
band = 1.96 * np.sqrt(2.0 / len(z_series)) / (2.0 * bias)
print("by hand B", round(bias, 4), "band +/-", round(band, 4))
stats = eval_risk.bias_statistics(z_series)
assert abs(bias - stats["bias"]) < 1e-12
assert abs((stats["bias_upper"] - stats["bias_lower"]) / 2 - band) < 1e-6

by hand z -0.984666 stored z -0.984666
by hand B 1.0931 band +/- 0.0217


## 3. By hand: the coverage share and the Q-Q pair

In [5]:
# coverage = share of |z| above 1.96 against the nominal five percent, and the
# Q-Q pair is the OLS of the empirical quantiles on the standard normal
coverage = float(np.mean(np.abs(z_series) > eval_risk.Z_CRITICAL))
probabilities = np.linspace(0, 1, len(z_series) + 2)[1:-1]
slope, intercept = np.polyfit(
    eval_risk.norm_ppf(probabilities), np.quantile(z_series, probabilities), 1
)
print("by hand coverage", round(coverage, 4), "qq", round(slope, 4), round(intercept, 4))
summary = pd.read_parquet(DATA / "eval" / "e5_bias_summary.parquet")
row = summary.loc[
    (summary["version"] == "xs_v1")
    & (summary["family"] == "long_only")
    & (summary["portfolio"] == portfolio)
].iloc[0]
assert abs(coverage - float(row["coverage"])) < 1e-9
assert abs(slope - float(row["qq_slope"])) < 1e-9
assert abs(intercept - float(row["qq_intercept"])) < 1e-9

by hand coverage 0.0543 qq 0.977 0.0652


## 4. By hand: the champion rule's arithmetic on the stored table

In [6]:
# the rule, verbatim, applied to the stored family-pooled table: mean |bias-1|
# across families, ties to fewer parameters, eligible versions only
print("rule:", REGISTRY["champion_rule"])
family = pd.read_parquet(DATA / "eval" / "e5_family_bias.parquet")
eligible = registry.eligible(REGISTRY)
scores = {v: float(family.loc[family["version"] == registry.engine_tag(v), "abs_bias_minus_1"].mean()) for v in eligible}
parameters = {
    v: float(REGISTRY["models"][v]["parameters"].get("n_factors", REGISTRY["models"][v]["parameters"].get("k_mean", 0.0)) or 0.0)
    + float(REGISTRY["models"][v]["parameters"].get("n_names", REGISTRY["models"][v]["parameters"].get("n_names_mean", 0.0)) or 0.0)
    for v in eligible
}
winner = min(scores, key=scores.get)
band = float(family.loc[family["version"] == registry.engine_tag(winner), "bias_se"].max())
near = [v for v in scores if abs(scores[v] - scores[winner]) < band]
if len(near) > 1:
    winner = min(near, key=parameters.get)
print(pd.Series(scores).sort_values().round(4).to_string())
print("tie candidates inside the band:", near)
print("winner:", winner, "| registry champion:", registry.champion(REGISTRY))
assert winner == registry.champion(REGISTRY), "the arithmetic disagrees with the registry"

rule: min mean |bias-1| across portfolio families; ties to fewer parameters; a champion must be refreshable daily from EFB's own data
XS-v1      0.0607
XS-v2      0.0672
PCA-v1     0.1388
PCA-v1c    0.1392
tie candidates inside the band: ['XS-v1']
winner: XS-v1 | registry champion: XS-v1


## 5. The D4 panel-to-column map, and the non-empty guard

In [7]:
from dashboard.tabs import d04_risk_eval as d4  # noqa: E402

panel_map = {
    name: list(frame.columns)
    for name, frame in {
        "bias_heatmap_panel": d4.bias_heatmap_panel(),
        "rolling_bias_panel": d4.rolling_bias_panel(),
        "calibration_panel": d4.calibration_panel(),
        "horizon_panel": d4.horizon_panel(),
    }.items()
}
for name, columns in panel_map.items():
    print(name, "->", columns)
badge = d4.champion_badge()
print("badge:", badge["champion"], "rule quoted:", badge["rule"] == REGISTRY["champion_rule"])
try:
    d4._require(pd.DataFrame(), "empty probe")
    raise AssertionError("the guard did not fire")
except ValueError:
    print("the non-empty guard raises on an empty read")

bias_heatmap_panel -> ['version', 'family', 'regime', 'bias']
rolling_bias_panel -> ['version', 'family', 'date', 'rolling_bias', 'rolling_bias_lower', 'rolling_bias_upper', 'n_pairs']
calibration_panel -> ['version', 'family', 'bias', 'bias_lower', 'bias_upper', 'coverage', 'mad_ratio', 'qq_slope', 'qq_intercept']
horizon_panel -> ['version', 'family', 'n_obs', 'bias_scaled', 'bias_direct', 'ratio']
badge: XS-v1 rule quoted: True
the non-empty guard raises on an empty read


## 6. Evidence for the deliverable, in citation order

In [8]:
# the numbers the memo cites, in the order it cites them
print("F5.1 family bias means:", STORED["F5.1"]["family_bias"])
print("F5.2 distances:", STORED["F5.2"]["long_short_abs_bias_minus_1"])
print("F5.2 sample distance:", STORED["F5.2"]["sample_long_short_abs_bias_minus_1"])
print("F5.3 2020 Q1:", STORED["F5.3"]["bias_2020_q1"])
print("F5.3 2022:", STORED["F5.3"]["bias_2022"])
print("F5.4 champion stress and recovery:", STORED["F5.4"])
print("F5.5 ratios:", STORED["F5.5"]["ratio_xs_v2_over_xs_v1"])
haircut = pd.read_parquet(DATA / "eval" / "e5_stress_haircut.parquet").iloc[0]
print("haircut:", round(float(haircut["stress_haircut"]), 4))
print("haircut basis:", haircut["basis"])
horizon = pd.read_parquet(DATA / "eval" / "e5_horizon.parquet")
print(horizon.loc[horizon["version"] == "xs_v1"].round(4).to_string(index=False))

F5.1 family bias means: {'XS-v1': 1.0469839503253322, 'PCA-v1': 1.1388412938997308, 'PCA-v1c': 1.1391854728617519, 'XS-v2': 1.0384709489361823}
F5.2 distances: {'ts_v1': 0.06632785175621136, 'xs_v1': 0.018568115349799186, 'xs_v2': 0.02563794967801525, 'pca_v1': 0.07288545487816167, 'pca_v1c': 0.07545776635627166}
F5.2 sample distance: 0.07522700661959725
F5.3 2020 Q1: {'pca_v1': 3.2235857142567585, 'pca_v1c': 3.2320487695936655, 'sample': 3.205007104643809, 'ts_v1': 3.498892154914531, 'xs_v1': 2.8470668684488754, 'xs_v2': 2.830534414326818}
F5.3 2022: {'pca_v1': 1.1992676970117708, 'pca_v1c': 1.201279826475275, 'sample': 1.1892311574499652, 'ts_v1': 1.3552221759980636, 'xs_v1': 1.139278188172363, 'xs_v2': 1.1291929280554092}
F5.4 champion stress and recovery: {'n_versions_in_regime_table': 6, 'champion': 'XS-v1', 'champion_stress_bias_vix_high': 1.3238537403338062, 'champion_recovery_trading_days_2020_q1': 13.0}
F5.5 ratios: {'factor_tilted': 0.9690679080256445, 'long_only': 0.99976997

## 7. What E6 inherits

In [9]:
for name, entry in sorted(REGISTRY["models"].items()):
    print(
        name,
        "champion" if entry["champion"] else "",
        "eligible" if entry["eligible_for_champion"] else "diagnostic only",
    )
print("open questions E6 inherits:")
print("  1. F5.1 fails: no version is calibrated on the long-only family.")
print("  2. the sqrt(21) scaled and the direct 21-day model disagree, stored per family.")
print("  3. the survivor-only universe and the pre-2015-10-23 look-ahead caveat.")
print("  4. the stress haircut 1.8471 is the working number until a stress model replaces it.")
print("  5. F5.0b: the covariance race runs eight estimators; the XS-v1 row is closed.")

PCA-v1  eligible
PCA-v1c  eligible
TS-v1  diagnostic only
XS-v1 champion eligible
XS-v2  eligible
open questions E6 inherits:
  1. F5.1 fails: no version is calibrated on the long-only family.
  2. the sqrt(21) scaled and the direct 21-day model disagree, stored per family.
  3. the survivor-only universe and the pre-2015-10-23 look-ahead caveat.
  4. the stress haircut 1.8471 is the working number until a stress model replaces it.
  5. F5.0b: the covariance race runs eight estimators; the XS-v1 row is closed.


## 8. Credit port note: what changes when the cross-section is bonds

In [10]:
n_sectors = int(pd.read_parquet(DATA / "processed" / "sectors.parquet")["gics_sector"].nunique())
print(f"the equity panel carries {n_sectors} GICS sectors and sqrt(mcap) weights.")
print("bonds: sector dummies become credit-sector buckets, sqrt(mcap) becomes")
print("spread-based sizing, the VIX terciles become spread-level terciles and")
print("2020 Q1 becomes the March 2020 spread widening; the bias statistic,")
print("the champion rule and the haircut basis transfer unchanged.")

the equity panel carries 11 GICS sectors and sqrt(mcap) weights.
bonds: sector dummies become credit-sector buckets, sqrt(mcap) becomes
spread-based sizing, the VIX terciles become spread-level terciles and
2020 Q1 becomes the March 2020 spread widening; the bias statistic,
the champion rule and the haircut basis transfer unchanged.


## 9. Closing checklist

In [11]:
def numeric_leaves(node):
    out = []
    if isinstance(node, dict):
        for value in node.values():
            out.extend(numeric_leaves(value))
    elif isinstance(node, list):
        for value in node:
            out.extend(numeric_leaves(value))
    elif isinstance(node, float):
        out.append(node)
    return out


notebook = json.loads((ROOT / "notebooks" / "E5_walkthrough.ipynb").read_text())
source = "\n".join(
    "".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code"
)
checked = numeric_leaves(RESULTS["criteria"]) + numeric_leaves(REGISTRY["models"])
offenders = sorted(
    {
        text
        for value in checked
        for text in (f"{value:.6f}", f"{value:.4f}")
        if len(text) > 6 and text in source
    }
)
assert not offenders, f"stored values typed into a code cell: {offenders}"
print("closing checklist")
print(f"  criteria evaluated:        {len(RESULTS['criteria'])}")
print(f"  verdicts:                  {sorted(set(VERDICTS.values()))}")
print(f"  data hash asserted:        {RESULTS['data_hash']}")
print(f"  champion declared:         {registry.champion(REGISTRY)}")
print(f"  stored values scanned:     {len(checked)}")
print(f"  literals found in cells:   {len(offenders)}")
print("  the standardized return, the bias band, the coverage, the Q-Q pair")
print("  and the champion arithmetic are all recomputed above from stored")
print("  artifacts and asserted against them.")

closing checklist
  criteria evaluated:        5
  verdicts:                  ['fail', 'pass']
  data hash asserted:        210769d6ec92a325ec2f5553875ea4de0de29bf52051a527f69ab674c098b293
  champion declared:         XS-v1
  stored values scanned:     97
  literals found in cells:   0
  the standardized return, the bias band, the coverage, the Q-Q pair
  and the champion arithmetic are all recomputed above from stored
  artifacts and asserted against them.
